In [ ]:
! pip install bert-for-sequence-classification==0.0.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvj

In [ ]:
import os
import pandas as pd
import torch
import torch.optim as optim
import torch.nn as nn
import json

from transformers import AutoModel, AutoTokenizer

from bert_clf import BertCLF, train_evaluate, predict_metrics, prepare_data_notebook, prepare_dataset
from bert_clf.utils import set_global_seed

In [ ]:
# Import necessary modules
from bert_clf.src.early_stopping import EarlyStopping
import numpy as np

# Monkey-patch the EarlyStopping class
def _init__(self, config):  # Rename __init__ to _init__ to avoid recursion
    self.patience = config['training']['patience']
    self.counter = 0
    self.best_score = None
    self.early_stop = False
    self.val_loss_min = np.inf  # Use np.inf instead of np.Inf
    self.delta = config['training']['delta']
    self.config = config

# Assign the modified _init__ method to the original __init__ attribute
EarlyStopping.__init__ = _init__


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split



### Prepare UC-UNSC dataset

In [ ]:
df = pd.read_csv('sentence_CP.csv', sep =',')

In [ ]:
df

,Sentence_ID,Sentence,Arg_Type,Argument_IDs,Argument_Relations
0,1,"Once again, since the last briefing to the Cou...",claim,[1],{}
1,2,This is now the tenth time that the Council ha...,non-arg,[],{}
2,3,The General Assembly also took up the matter o...,non-arg,[],{}
3,4,"Following close to two weeks of relative calm,...",premise,[2],{}
4,5,The individuals involved called for secession ...,premise,[3],{}
...,...,...,...,...,...
4760,4761,We welcome Italy's decision to designate\ndial...,claim,[4556],{}
4761,4762,China supports practical and effective coopera...,claim,[4557],{}
4762,4763,We welcome all the positive efforts being made...,claim,[4558],{}
4763,4764,We hope that\nall the parties concerned will w...,claim,[4559],{}


In [ ]:
df['Label'] = df['Arg_Type'].apply(lambda x: 0 if x == 'non-arg' else 1)

In [ ]:
df['Label'].value_counts()

,count
Label,
1,4105
0,660


In [ ]:
df_train, df_temp = train_test_split(df, test_size=0.3, random_state=13)

df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=13)

In [ ]:
df_train

,Sentence_ID,Sentence,Arg_Type,Argument_IDs,Argument_Relations,Label
2016,2017,"""We control the border and\nwe will not let Uk...",premise,[1882],{},1
2832,2833,"By now, it is\nbarely making the world's headl...",premise,"[2676, 2677, 2678]","{(2676, 2678): 'support', (2676, 2677): 'suppo...",1
874,875,In order to implement the Geneva statement for...,claim,[823],{},1
4427,4428,We continue to be concerned about threats made...,premise,[4241],{},1
1313,1314,"Otherwise, the situation will further escalate...",premise,[1239],{},1
...,...,...,...,...,...,...
153,154,We call on all parties to exercise restraint a...,claim,[163],{},1
866,867,The onus is now on Russia to stop interfering ...,claim,[817],{},1
2790,2791,With the signing of the ceasefire in Minsk on\...,premise,[2636],{},1
74,75,Ukraine's Government has reached out repeatedl...,premise,[69],{},1


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X = df["Sentence"].values
y = df["Label"].values

all_reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n======== Fold {fold + 1} ========")

    df_train_fold = df.iloc[train_idx].reset_index(drop=True)
    df_val_fold = df.iloc[val_idx].reset_index(drop=True)

    # Set global seed per fold if needed
    set_global_seed(config['data']['random_state'])

    # Tokenizer and model must be re-initialized per fold
    tokenizer = AutoTokenizer.from_pretrained(config['transformer_model']["model"])
    model_bert = AutoModel.from_pretrained(config['transformer_model']["model"]).to(device)

    id2label, train_texts, valid_texts, train_targets, valid_targets = prepare_data_notebook(
        config=config,
        train_df=df_train_fold,
        test_df=df_val_fold
    )

    model = BertCLF(
        pretrained_model=model_bert,
        tokenizer=tokenizer,
        id2label=id2label,
        dropout=config['transformer_model']['dropout'],
        device=device
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=config['transformer_model']['learning_rate'])
    criterion = nn.NLLLoss()

    training_generator, valid_generator = prepare_dataset(
        tokenizer=tokenizer,
        train_texts=train_texts,
        train_targets=train_targets,
        valid_texts=valid_texts,
        valid_targets=valid_targets,
        config=config
    )

    model = train_evaluate(
        model=model,
        training_generator=training_generator,
        valid_generator=valid_generator,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=config['training']['num_epochs'],
        average=config['training']['average_f1'],
        config=config
    )

    # Evaluate
    model.to('cpu')
    preds = [model.predict(sent) for sent in df_val_fold['Sentence']]
    report = classification_report(df_val_fold['Label'], preds, output_dict=True)
    all_reports.append(report)

# Optional: Aggregate metrics
import numpy as np
macro_f1s = [r['macro avg']['f1-score'] for r in all_reports]
print(f"\nAverage Macro F1 over 5 folds: {np.mean(macro_f1s):.3f}")



======== Fold 1 ========


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/bert_clf/src/pandas_dataset/SimpleDataFrame.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.train.dropna(inplace=True)


==== Epoch 1 out of 6 ====


Evaluating loop: 100%|██████████| 60/60 [00:01<00:00, 32.03it/s]


Train F1: 0.47121984919475246
Eval F1: 0.5033966510175368

Train F1 micro: 0.7447698744769874
Eval F1 micro: 0.8616898148148148

Train F1 weighted: 0.7082870387601166
Eval F1 weighted: 0.799919660927889

==== Epoch 2 out of 6 ====


...

Train F1: 0.8106155191109377
Eval F1: 0.740764456705747

Train F1 micro: 0.9375
Eval F1 micro: 0.8950231481481481

Train F1 weighted: 0.929405718187652
Eval F1 weighted: 0.8920744205487421

==== Epoch 6 out of 6 ====


Evaluating loop: 100%|██████████| 60/60 [00:01<00:00, 32.92it/s]


Train F1: 0.8404951317471545
Eval F1: 0.8112670258098511

Train F1 micro: 0.9424686192468619
Eval F1 micro: 0.9144675925925926

Train F1 weighted: 0.9367158111279078
Eval F1 weighted: 0.90270369039031




Computing final metrics...: 100%|██████████| 60/60 [00:01<00:00, 39.97it/s]


              precision    recall  f1-score   support

           0       0.80      0.53      0.64       132
           1       0.93      0.98      0.95       821

    accuracy                           0.92       953
   macro avg       0.87      0.75      0.80       953
weighted avg       0.91      0.92      0.91       953


Average Macro F1 over 5 folds: 0.819


In [ ]:
macro_f1s = [r['0']['f1-score'] for r in all_reports]
print(f"\nAverage F1 for 0 over 5 folds: {np.mean(macro_f1s):.3f}")


Average F1 for 0 over 5 folds: 0.682


In [ ]:
import numpy as np

# Example structure: list of classification reports as dicts, e.g. from sklearn.metrics.classification_report(output_dict=True)
# all_reports = [report_fold1, report_fold2, ..., report_foldN]
# Each report looks like:
# {
#   'class_0': {'precision': ..., 'recall': ..., 'f1-score': ..., 'support': ...},
#   'class_1': {...},
#   'accuracy': ...,
#   'macro avg': {'precision': ..., 'recall': ..., 'f1-score': ..., 'support': ...},
#   'weighted avg': {...},
# }

classes = ['1', '0']  # Replace with your actual class labels, e.g. ['Claim', 'Premise']

metrics = ['precision', 'recall', 'f1-score']

# Compute mean and std for each class and metric
for cls in classes:
    print(f"\nResults for {cls}:")
    for metric in metrics:
        scores = [report[cls][metric] for report in all_reports]
        mean = np.mean(scores)
        std = np.std(scores)
        print(f"  {metric.capitalize():<9}: {mean:.3f} ± {std:.3f}")

# Macro-average (all classes)
print(f"\nMacro-averaged results over {len(all_reports)} folds:")
for metric in metrics:
    scores = [report['macro avg'][metric] for report in all_reports]
    mean = np.mean(scores)
    std = np.std(scores)
    print(f"  {metric.capitalize():<9}: {mean:.3f} ± {std:.3f}")

# (Optional) Weighted-average
print(f"\nWeighted-averaged results over {len(all_reports)} folds:")
for metric in metrics:
    scores = [report['weighted avg'][metric] for report in all_reports]
    mean = np.mean(scores)
    std = np.std(scores)
    print(f"  {metric.capitalize():<9}: {mean:.3f} ± {std:.3f}")



Results for 1:
  Precision: 0.937 ± 0.008
  Recall   : 0.978 ± 0.005
  F1-score : 0.957 ± 0.004

Results for 0:
  Precision: 0.810 ± 0.029
  Recall   : 0.591 ± 0.056
  F1-score : 0.682 ± 0.039

Macro-averaged results over 5 folds:
  Precision: 0.874 ± 0.015
  Recall   : 0.784 ± 0.027
  F1-score : 0.819 ± 0.021

Weighted-averaged results over 5 folds:
  Precision: 0.919 ± 0.008
  Recall   : 0.924 ± 0.007
  F1-score : 0.919 ± 0.009


In [ ]:
macro_f1s = [r['0']['f1-score'] for r in all_reports]
print(f"\nAverage F1 for 0 over 5 folds: {np.mean(macro_f1s):.3f}")


Average F1 for 0 over 5 folds: 0.682


In [ ]:
import numpy as np

macro_f1s = [r['0']['f1-score'] for r in all_reports]
mean_f1 = np.mean(macro_f1s)
std_f1 = np.std(macro_f1s)

print(f"\nAverage F1 for class 0 over 5 folds: {mean_f1:.3f} ± {std_f1:.3f}")


Average F1 for class 0 over 5 folds: 0.682 ± 0.039


In [ ]:
macro_f1s = [r['1']['f1-score'] for r in all_reports]
mean_f1 = np.mean(macro_f1s)
std_f1 = np.std(macro_f1s)

print(f"\nAverage F1 for class 1 over 5 folds: {mean_f1:.3f} ± {std_f1:.3f}")


Average F1 for class 1 over 5 folds: 0.957 ± 0.004


In [ ]:
macro_f1s = [r['macro avg']['f1-score'] for r in all_reports]
mean_f1 = np.mean(macro_f1s)
std_f1 = np.std(macro_f1s)

print(f"\nAverage F1 for class 0 over 5 folds: {mean_f1:.3f} ± {std_f1:.3f}")


Average F1 for class 0 over 5 folds: 0.819 ± 0.021


In [ ]:
# prelim verdict -- seems that 6 epochs is fine here, make things better.

### Transformer Language Model

In [ ]:
config = dict(
    transformer_model = dict(
        model = "roberta-base",
        path_to_state_dict = False,
        device = 'cuda',
        dropout = 0.2,
        learning_rate = 2e-6,
        batch_size = 16,
        shuffle = True,
        maxlen = 128,
    ),
    data = dict(
        train_data_path = df_train,
        test_data_path = df_val,
        text_column = "Sentence",
        target_column = "Label",
        random_state = 20,
        test_size = 0.3,
        stratify=True
    ),
    training = dict (
    save_state_dict = False, # if False the model will be saved using torch.save(<model_class>)
        # and should be loaded like this: model = torch.load()
        # you will have to install the library to do so
    early_stopping = True,
    delta = 0.001,
    patience = 7,
    num_epochs = 6,
    average_f1 = 'macro',
    other_metrics = ['micro', 'weighted'],
    output_dir = "../results/",
    class_weight = True
    )
)

In [ ]:
set_global_seed(seed=config['data']['random_state'])
os.makedirs(config['training']['output_dir'], exist_ok=True)

In [ ]:
device = torch.device(config['transformer_model']['device'])
tokenizer = AutoTokenizer.from_pretrained(
        pretrained_model_name_or_path=config['transformer_model']["model"]
    )
model_bert = AutoModel.from_pretrained(
    pretrained_model_name_or_path=config['transformer_model']["model"]
).to(device)

#for param in model_bert.parameters():
    #param.requires_grad = False

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
id2label, train_texts, valid_texts, train_targets, valid_targets = prepare_data_notebook(
    config=config, train_df = df_train, test_df = df_val
)

/usr/local/lib/python3.11/dist-packages/bert_clf/src/pandas_dataset/SimpleDataFrame.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.train.dropna(inplace=True)


In [ ]:
model = BertCLF(
    pretrained_model=model_bert,
    tokenizer=tokenizer,
    id2label=id2label,
    dropout=config['transformer_model']['dropout'],
    device=device
    )

In [ ]:
model = model.to(device)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=float(config['transformer_model']['learning_rate']))
criterion = nn.NLLLoss()

training_generator, valid_generator = prepare_dataset(
    tokenizer=tokenizer,
    train_texts=train_texts,
    train_targets=train_targets,
    valid_texts=valid_texts,
    valid_targets=valid_targets,
    config=config
)

In [ ]:
# takes 2 min
model = train_evaluate(
    model=model,
    training_generator=training_generator,
    valid_generator=valid_generator,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=config['training']['num_epochs'],
    average=config['training']['average_f1'],
    config=config
)

==== Epoch 1 out of 2 ====


Evaluating loop: 100%|██████████| 45/45 [00:04<00:00, 10.86it/s]


Train F1: 0.47343500193576915
Eval F1: 0.48344998515311205

Train F1 micro: 0.7413277511961722
Eval F1 micro: 0.8597222222222223

Train F1 weighted: 0.7084617399153312
Eval F1 weighted: 0.7969889185826649

==== Epoch 2 out of 2 ====


Evaluating loop: 100%|██████████| 45/45 [00:04<00:00,  9.60it/s]


Train F1: 0.6623041117314694
Eval F1: 0.7278414309110145

Train F1 micro: 0.8892686261107314
Eval F1 micro: 0.9069444444444444

Train F1 weighted: 0.8572328221203434
Eval F1 weighted: 0.8797747500237304




Computing final metrics...: 100%|██████████| 45/45 [00:04<00:00, 10.58it/s]

              precision    recall  f1-score   support

           0       0.95      0.36      0.52       101
           1       0.90      1.00      0.95       614

    accuracy                           0.91       715
   macro avg       0.93      0.68      0.73       715
weighted avg       0.91      0.91      0.89       715



In [ ]:
model.to('cpu')

BertCLF(
  (pretrained_model): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm)

In [ ]:
# takes 2-3 min.

preds = []
for i,j in zip(df_test['Sentence'], df_test['Label']):
    preds.append([model.predict(i), j, i])

In [ ]:
pred = []
for i in preds:
    pred.append(i[0])

true = []
for m in preds:
    true.append(m[1])

In [ ]:
# second
target_names = ['class 0', 'class 1']
print(classification_report(true, pred, target_names=target_names, digits=3))

              precision    recall  f1-score   support

     class 0      1.000     0.356     0.525        87
     class 1      0.918     1.000     0.957       628

    accuracy                          0.922       715
   macro avg      0.959     0.678     0.741       715
weighted avg      0.928     0.922     0.905       715



In [ ]:
# first
from sklearn.metrics import classification_report

target_names = ['class 0', 'class 1']
print(classification_report(true, pred, target_names=target_names, digits=3))

              precision    recall  f1-score   support

     class 0      0.575     0.548     0.561        84
     class 1      0.940     0.946     0.943       631

    accuracy                          0.899       715
   macro avg      0.758     0.747     0.752       715
weighted avg      0.897     0.899     0.898       715



In [ ]:
dummy = np.ones(715)

In [ ]:
# baseline for task 1 on UN-UNSC
target_names = ['class 0', 'class 1']
print(classification_report(true, dummy, target_names=target_names, digits=3))

              precision    recall  f1-score   support

     class 0      0.000     0.000     0.000        87
     class 1      0.878     1.000     0.935       628

    accuracy                          0.878       715
   macro avg      0.439     0.500     0.468       715
weighted avg      0.771     0.878     0.821       715



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
